**Instituto Brasileiro de Ensino, Desenvolvimento e Pesquisa**

Programa de Pós-Graduação em Administração Pública — Mestrado Profissional

**Disciplina:** Avaliação de Políticas Públicas com Dados  
**Atividade:** Trabalho final em grupo — análise causal do Bolsa Família  
**Grupo:** Analécia Borato, Fabricio Santana, Giovanni Brígido, Paulo Ceser

# Bolsa Família e evasão escolar

## Trabalho final — análise causal com ATT e diferenças em diferenças

Este notebook estima o efeito do Bolsa Família sobre a evasão escolar por duas estratégias complementares. O ATT usa seleção nos observáveis; o DiD usa o painel domiciliar de 2005 e 2009. As amostras e os estimandos são diferentes e, portanto, os resultados são interpretados como triangulação.

In [2]:
from pathlib import Path
individual_path = Path("data/aula03_completo.csv")
panel_path = Path("data/aula04_painel.csv")
assert individual_path.exists(), individual_path
assert panel_path.exists(), panel_path
print(individual_path)
print(panel_path)

data/aula03_completo.csv
data/aula04_painel.csv


In [3]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.calibration import calibration_curve, CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
import statsmodels.formula.api as smf
from linearmodels.panel import PanelOLS

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
sns.set_theme(style="whitegrid", context="notebook")
RANDOM_STATE = 42

## 1. Dados e recorte do ATT

A unidade é a pessoa de 6 a 17 anos. O tratamento é domiciliar: há tratamento quando pelo menos uma pessoa do domicílio possui titularidade do cartão. O grupo controle é C1.

In [4]:
aibf = pd.read_csv(individual_path)
aibf["D"] = aibf.groupby("cod_dtm")["s02a10"].transform(lambda s: int((s == 1).any()))
aibf["feminino"] = (aibf["s02ad"] == 2).astype(float)
aibf["agua"] = (aibf["agua_canalizada"] == 1).astype(float)
COVARIAVEIS = ["s02af", "feminino", "educ_chefe", "n_moradores", "n_comodos", "n_dormitorios", "agua"]
ROTULOS = {"s02af": "Idade", "feminino": "Sexo feminino", "educ_chefe": "Escolaridade do chefe", "n_moradores": "Moradores", "n_comodos": "Cômodos", "n_dormitorios": "Dormitórios", "agua": "Água canalizada"}
filtro = (aibf["estrato_amostral"].isin(["beneficiario", "controle"]) & aibf["s02af"].between(6, 17) & aibf["dropout"].notna())
dados = (aibf.loc[filtro, ["cod_dtm", "D", "dropout", *COVARIAVEIS]].dropna().rename(columns={"dropout": "Y"}).reset_index(drop=True))
assert set(dados["D"].unique()) == {0, 1}
print(f"Base bruta: {len(aibf):,} pessoas, {aibf.cod_dtm.nunique():,} domicílios")
print(f"Amostra ATT: {len(dados):,} pessoas, {dados.cod_dtm.nunique():,} domicílios")
resumo = dados.groupby("D")["Y"].agg(n="size", evasao="mean").rename(index={0: "C1 controle", 1: "Beneficiário"})
resumo["evasao_p.p."] = 100 * resumo["evasao"]
resumo

Base bruta: 56,367 pessoas, 11,372 domicílios
Amostra ATT: 8,484 pessoas, 4,656 domicílios


,n,evasao,evasao_p.p.
D,,,
C1 controle,2430,0.0786,7.8601
Beneficiário,6054,0.0689,6.8880


## 2. ATT por escore de propensão

O IPW-Hajek preserva a distribuição dos tratados e repondera controles. O pareamento usa o controle com escore mais próximo, com reposição. O suporte comum e o balanceamento são diagnosticados antes da interpretação.

In [5]:
X = dados[COVARIAVEIS]
D = dados["D"].to_numpy().astype(int)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
logit = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, random_state=RANDOM_STATE))
rf = CalibratedClassifierCV(RandomForestClassifier(n_estimators=300, min_samples_leaf=25, max_features="sqrt", n_jobs=-1, random_state=RANDOM_STATE), method="sigmoid", cv=3)
dados["ps_logit"] = cross_val_predict(logit, X, D, cv=cv, method="predict_proba", n_jobs=-1)[:, 1]
dados["ps_rf"] = cross_val_predict(rf, X, D, cv=cv, method="predict_proba", n_jobs=-1)[:, 1]
pd.DataFrame({"modelo": ["Logística", "Random forest"], "AUC": [roc_auc_score(D, dados.ps_logit), roc_auc_score(D, dados.ps_rf)], "mínimo": [dados.ps_logit.min(), dados.ps_rf.min()], "máximo": [dados.ps_logit.max(), dados.ps_rf.max()]}).round(3)

,modelo,AUC,mínimo,máximo
0,Logística,0.6070,0.3200,0.9240
1,Random forest,0.6130,0.4430,0.8460


In [6]:
def suporte_comum(df, ps_col):
    limites = df.groupby("D")[ps_col].agg(["min", "max"])
    inferior = limites["min"].max(); superior = limites["max"].min()
    return df[ps_col].between(inferior, superior), inferior, superior

def ess(w):
    w = np.asarray(w); return w.sum() ** 2 / (w ** 2).sum()

def inferencia_cluster(df, pesos):
    d = df.D.to_numpy().astype(bool); y = df.Y.to_numpy(); w = np.asarray(pesos)
    mu1 = np.average(y[d], weights=w[d]); mu0 = np.average(y[~d], weights=w[~d])
    influencia = np.zeros(len(df)); influencia[d] = w[d] * (y[d] - mu1) / w[d].sum(); influencia[~d] = -w[~d] * (y[~d] - mu0) / w[~d].sum()
    por_dom = pd.Series(influencia).groupby(df.cod_dtm.reset_index(drop=True)).sum(); g = len(por_dom)
    return np.sqrt((g / (g - 1)) * np.square(por_dom).sum())

def estimar_att(df, ps_col):
    keep, inferior, superior = suporte_comum(df, ps_col); sub = df.loc[keep].copy()
    treated = sub.D.eq(1).to_numpy(); ps = sub[ps_col].to_numpy(); y = sub.Y.to_numpy()
    it = np.flatnonzero(treated); ic = np.flatnonzero(~treated)
    nn = NearestNeighbors(n_neighbors=1).fit(ps[ic, None]); dist, viz = nn.kneighbors(ps[it, None]); pares = ic[viz.ravel()]
    w_match = np.zeros(len(sub)); w_match[it] = 1; np.add.at(w_match, pares, 1)
    w_ipw = np.where(treated, 1.0, ps / (1 - ps))
    att_match = np.mean(y[it] - y[pares]); att_ipw = np.average(y[treated], weights=w_ipw[treated]) - np.average(y[~treated], weights=w_ipw[~treated])
    return sub, w_match, w_ipw, att_match, att_ipw, (inferior, superior), dist.ravel()

att_rows = []; ajustes = {}
for nome, coluna in [("Logística", "ps_logit"), ("Random forest", "ps_rf")]:
    sub, wm, wi, am, ai, suporte, dist = estimar_att(dados, coluna); ajustes[nome] = (sub, wm, wi)
    for metodo, att, w in [("Pareamento 1:1", am, wm), ("IPW-ATT", ai, wi)]:
        ep = inferencia_cluster(sub, w)
        att_rows.append({"modelo": nome, "método": metodo, "ATT_p.p.": 100 * att, "IC95_inf": 100 * (att - 1.96 * ep), "IC95_sup": 100 * (att + 1.96 * ep), "suporte": str(tuple(round(x, 3) for x in suporte))})
resultados_att = pd.DataFrame(att_rows)
resultados_att.round(3)

,modelo,método,ATT_p.p.,IC95_inf,IC95_sup,suporte
0,Logística,Pareamento 1:1,-1.5530,-3.5320,0.4260,"(np.float64(0.372), np.float64(0.923))"
1,Logística,IPW-ATT,-1.1070,-2.6220,0.4090,"(np.float64(0.372), np.float64(0.923))"
2,Random forest,Pareamento 1:1,-2.3300,-4.3730,-0.2870,"(np.float64(0.455), np.float64(0.843))"
3,Random forest,IPW-ATT,-1.1150,-2.5820,0.3520,"(np.float64(0.455), np.float64(0.843))"


In [7]:
def smd(df, pesos=None):
    d = df.D.to_numpy().astype(bool); w = np.ones(len(df)) if pesos is None else np.asarray(pesos); linhas = []
    for var in COVARIAVEIS:
        x = df[var].to_numpy(); m1 = np.average(x[d], weights=w[d]); m0 = np.average(x[~d], weights=w[~d])
        sd = np.sqrt((np.var(x[d], ddof=1) + np.var(x[~d], ddof=1)) / 2)
        linhas.append({"covariável": ROTULOS[var], "SMD": (m1 - m0) / sd})
    return pd.DataFrame(linhas)

antes = smd(dados).rename(columns={"SMD": "Antes"})
balanceamento = []
for nome, (sub, wm, wi) in ajustes.items():
    t = antes.copy(); t["Pareamento"] = smd(sub, wm)["SMD"]; t["IPW"] = smd(sub, wi)["SMD"]; t.insert(0, "modelo", nome); balanceamento.append(t)
balanceamento = pd.concat(balanceamento, ignore_index=True)
balanceamento.groupby("modelo")[["Antes", "Pareamento", "IPW"]].apply(lambda x: x.abs().max()).round(3)

,Antes,Pareamento,IPW
modelo,,,
Logística,0.2570,0.0350,0.0410
Random forest,0.2570,0.0350,0.1080


## 3. Diferenças em diferenças

O DiD compara a mudança em T1 com a mudança em C1. O placebo C1 × C2 é apresentado como diagnóstico, não como prova de tendências paralelas.

In [8]:
painel = pd.read_csv(panel_path)
assert not painel.duplicated(["cod_dtm", "ano"]).any()

def preparar(base, grupos, tratado):
    d = base[base.grupo.isin(grupos) & base.dropout_med.notna()].copy()
    d["tratamento"] = (d.grupo == tratado).astype(int); d["pos"] = (d.ano == 2009).astype(int)
    return d

def tabela_2x2(d):
    medias = d.groupby(["tratamento", "pos"])["dropout_med"].mean().unstack()
    did = (medias.loc[1, 1] - medias.loc[1, 0]) - (medias.loc[0, 1] - medias.loc[0, 0])
    return medias, did

def did_regressao(d):
    return smf.ols("dropout_med ~ tratamento + pos + tratamento:pos", data=d).fit(cov_type="cluster", cov_kwds={"groups": d.cod_dtm})

principal = preparar(painel, ["T1", "C1"], "T1")
placebo = preparar(painel, ["C1", "C2"], "C1")
for nome, d in [("T1 x C1", principal), ("Placebo C1 x C2", placebo)]:
    medias, did = tabela_2x2(d); reg = did_regressao(d)
    print(nome, "n domicílios =", d.cod_dtm.nunique(), "DiD =", round(100 * did, 3), "p.p.", "p =", round(reg.pvalues["tratamento:pos"], 3))
    display(medias)

T1 x C1 n domicílios = 6216 DiD = -0.798 p.p. p = 0.393


pos,0,1
tratamento,,
0,0.0388,0.0929
1,0.0312,0.0773


Placebo C1 x C2 n domicílios = 8604 DiD = 0.858 p.p. p = 0.341


pos,0,1
tratamento,,
0,0.0382,0.0837
1,0.0388,0.0929


In [9]:
def twfe(d):
    p = d.set_index(["cod_dtm", "ano"])[["dropout_med", "tratamento", "pos"]].copy()
    p["D"] = p["tratamento"] * p["pos"]
    return PanelOLS(p.dropout_med, p[["D"]], entity_effects=True, time_effects=True).fit(cov_type="clustered", cluster_entity=True, auto_df=False)

reg_principal = did_regressao(principal); reg_placebo = did_regressao(placebo)
res_did = []
for nome, d, reg in [("T1 x C1", principal, reg_principal), ("Placebo C1 x C2", placebo, reg_placebo)]:
    _, did = tabela_2x2(d); tw = twfe(d)
    res_did.extend([
        {"comparação": nome, "especificação": "DiD manual", "estimativa_p.p.": 100 * did, "p_valor": reg.pvalues["tratamento:pos"], "n_domicílios": d.cod_dtm.nunique()},
        {"comparação": nome, "especificação": "TWFE", "estimativa_p.p.": 100 * tw.params["D"], "p_valor": tw.pvalues["D"], "n_domicílios": d.cod_dtm.nunique()},
    ])
resultados_did = pd.DataFrame(res_did); resultados_did.round(3)

,comparação,especificação,estimativa_p.p.,p_valor,n_domicílios
0,T1 x C1,DiD manual,-0.7980,0.3930,6216
1,T1 x C1,TWFE,-0.5620,0.7430,6216
2,Placebo C1 x C2,DiD manual,0.8580,0.3410,8604
3,Placebo C1 x C2,TWFE,-0.4350,0.7980,8604


In [10]:
# Harmonização: cada comparação usa somente domicílios com os dois anos observados.
def preparar_balanceado(base, grupos, tratado):
    d = base[base.grupo.isin(grupos) & base.dropout_med.notna()].copy()
    completos = d.groupby("cod_dtm")["ano"].nunique()
    d = d[d.cod_dtm.isin(completos[completos == 2].index)].copy()
    d["tratamento"] = (d.grupo == tratado).astype(int); d["pos"] = (d.ano == 2009).astype(int)
    return d

principal = preparar_balanceado(painel, ["T1", "C1"], "T1")
placebo = preparar_balanceado(painel, ["C1", "C2"], "C1")
res_did = []
for nome, d in [("T1 x C1", principal), ("Placebo C1 x C2", placebo)]:
    _, did = tabela_2x2(d); reg = did_regressao(d); tw = twfe(d)
    res_did.extend([
        {"comparação": nome, "especificação": "DiD manual", "estimativa_p.p.": 100 * did, "p_valor": reg.pvalues["tratamento:pos"], "n_domicílios": d.cod_dtm.nunique()},
        {"comparação": nome, "especificação": "TWFE", "estimativa_p.p.": 100 * tw.params["D"], "p_valor": tw.pvalues["D"], "n_domicílios": d.cod_dtm.nunique()},
    ])
resultados_did = pd.DataFrame(res_did)
resultados_did.round(3)

,comparação,especificação,estimativa_p.p.,p_valor,n_domicílios
0,T1 x C1,DiD manual,-0.5620,0.5730,3201
1,T1 x C1,TWFE,-0.5620,0.6900,3201
2,Placebo C1 x C2,DiD manual,-0.4350,0.6520,4101
3,Placebo C1 x C2,TWFE,-0.4350,0.7500,4101


## 4. Síntese

As estimativas devem ser comparadas em pontos percentuais, sem afirmar que são o mesmo estimando. O notebook reproduz os diagnósticos e deixa registradas as limitações de identificação.

In [11]:
diferenca_bruta = 100 * (dados.loc[dados.D.eq(1), "Y"].mean() - dados.loc[dados.D.eq(0), "Y"].mean())
principal_did = resultados_did.query("comparação == 'T1 x C1' and especificação == 'DiD manual'").iloc[0]
ipw_logit = resultados_att.query("modelo == 'Logística' and método == 'IPW-ATT'").iloc[0]
sintese = pd.DataFrame([
    {"estratégia": "Diferença bruta", "estimativa_p.p.": diferenca_bruta, "IC95_inf": np.nan, "IC95_sup": np.nan},
    {"estratégia": "IPW-ATT (logística)", "estimativa_p.p.": ipw_logit["ATT_p.p."], "IC95_inf": ipw_logit["IC95_inf"], "IC95_sup": ipw_logit["IC95_sup"]},
    {"estratégia": "DiD T1 x C1", "estimativa_p.p.": principal_did["estimativa_p.p."], "IC95_inf": np.nan, "IC95_sup": np.nan},
])
sintese.round(3)

,estratégia,estimativa_p.p.,IC95_inf,IC95_sup
0,Diferença bruta,-0.9720,NaN,NaN
1,IPW-ATT (logística),-1.1070,-2.6220,0.4090
2,DiD T1 x C1,-0.5620,NaN,NaN


### Conclusão

Os resultados apontam para uma redução modesta da evasão entre beneficiários, mas os intervalos de confiança são amplos e incluem zero. A evidência é sugestiva, não conclusiva. O ATT requer ignorabilidade condicional; o DiD requer tendências paralelas, que não podem ser avaliadas com apenas um período pré.